# Bài học 18: Giao diện Chat

## Từ dòng lệnh đến hội thoại

Bài trước, bạn đã học cách dùng CLI để gõ lệnh. Nhưng nếu bạn **không nhớ** lệnh thì sao? Hoặc muốn **hỏi nhiều thứ** trong một cuộc trò chuyện?

**Giao diện chat** cho phép bạn nói chuyện tự nhiên với AI:

- *"Tạo một bài viết về on-page SEO"*
- *"Bài viết đó xong chưa?"*
- *"Cho tôi xem tất cả bài viết đang chờ duyệt"*

AI hiểu ý định của bạn và tự động thực hiện — không cần thuộc cú pháp lệnh.

## Agno Team — Một đội AI chuyên biệt

Giao diện chat được xây dựng với **Agno Team** — một nhóm các agent AI, mỗi agent có một vai trò riêng.

Cách hoạt động:
1. Bạn gửi một tin nhắn
2. **Leader** (trưởng nhóm) đọc và hiểu yêu cầu của bạn
3. Leader **giao việc** cho thành viên phù hợp
4. Thành viên thực hiện công việc (gọi tools) và trả kết quả
5. Leader **tổng hợp** và phản hồi lại cho bạn

Giống như một **quản lý dự án** thực thụ — bạn nói muốn gì, PM biết giao cho ai.

In [ ]:
print("""
SEO Workspace Team:
  
  Leader (Claude Opus) — Nhận yêu cầu, giao việc cho thành viên
    |
    +-- Content Creator (Claude Sonnet)
    |     Tools: create_article, create_article_batch
    |     Vai trò: Tạo bài viết mới
    |
    +-- Status Tracker (Claude Sonnet)
          Tools: list_all_articles, get_article_details, 
                 get_article_content, get_version_history
          Vai trò: Kiểm tra trạng thái bài viết

Ví dụ hội thoại:
  Bạn: "Tạo một bài viết về on-page SEO"
  -> Leader giao cho Content Creator
  -> Content Creator chạy pipeline
  -> Leader: "Đã tạo bài viết #5, 2000 từ, file: content/seo-on-page.md"

  Bạn: "Bài viết đó trạng thái sao rồi?"
  -> Leader giao cho Status Tracker
  -> Status Tracker kiểm tra database
  -> Leader: "Bài viết #5 đã hoàn thành, 2000 từ"
""")

## Workspace Tools

Các thành viên gọi **workspace tools** để thực hiện công việc. Đây là các hàm Python thuần túy trong `workspace_tools.py`:

| Tool | Thành viên | Mục đích |
|------|--------|--------|
| `create_article` | Content Creator | Tạo 1 bài viết (chạy pipeline) |
| `create_article_batch` | Content Creator | Tạo nhiều bài viết cùng lúc |
| `list_all_articles` | Status Tracker | Xem tất cả bài viết |
| `get_article_details` | Status Tracker | Xem chi tiết 1 bài viết |
| `get_article_content` | Status Tracker | Xem nội dung bài viết |
| `get_version_history` | Status Tracker | Xem lịch sử phiên bản |

Đây chính là những hàm bạn đã học trong các bài trước — chỉ được đóng gói lại để các thành viên trong team dùng làm tools.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

from workspace_tools import (
    create_article,
    create_article_batch,
    list_all_articles,
    get_article_details,
    get_article_content,
    get_version_history,
)

# Đây là các hàm Python thuần túy!
# Các thành viên trong team gọi chúng như tools

# Ví dụ: xem tất cả bài viết
import json
result = list_all_articles()
articles = json.loads(result)
print(f"Tìm thấy {len(articles)} bài viết trong workspace")
for a in articles:
    print(f"  #{a['id']}: {a['topic']} ({a['status']})")

## Chạy giao diện chat

Giao diện chat chạy trong terminal (không phải trong notebook). Khi chạy, bạn sẽ thấy:
- Lời chào và hướng dẫn sử dụng
- Dấu `>` để bạn nhập tin nhắn
- Phản hồi và hành động của AI
- Gõ `quit` hoặc `exit` để thoát

Lịch sử chat được lưu trong `chat_sessions.db`, nên lần sau bạn có thể tiếp tục từ chỗ dừng lại.

In [ ]:
print("Để chạy giao diện chat:")
print()
print("  python output/chat.py")
print()
print("Hoặc:")
print()
print("  python output/cli.py chat")
print()
print("Lịch sử chat được lưu trong chat_sessions.db")
print("nên bạn có thể tiếp tục từ chỗ dừng lại.")

## Đọc code sản phẩm

Bạn đã thấy mọi khái niệm qua các notebook. Code sản phẩm trong `output/` sử dụng **đúng những pattern đó** — đây là nơi tìm từng phần:

### Bản đồ: Bài học → File sản phẩm

| Bạn đã học | Bài học | File sản phẩm |
|---|---|---|
| Cách LLM hoạt động, prompt, model | 05-07 | (Kiến thức nền — định hướng mọi quyết định) |
| Tạo agent (name, model, instructions, tools) | 08-09 | `output/agents/builders.py` |
| Pydantic schema (ContentOutline, EnrichedContent) | 10, 13 | `output/agents/schemas.py` |
| Nối chuỗi agent, mini pipeline, pipeline đầy đủ | 11-12, 13-15 | `output/pipeline.py` |
| Custom toolkit (FreepikTools, DataForSEOTools) | 09, 14 | `output/tools/` |
| Database (SQLite, các hàm CRUD) | 16 | `output/db.py` |
| CLI (argparse, command handler) | 17 | `output/cli.py` |
| Agno Team (leader + member + workspace tools) | 18 | `output/chat.py` + `output/workspace_tools.py` |

### Cấu trúc file

```
output/
├── agents/                     # Các AI worker (bài 8-14)
│   ├── builders.py             # 4 định nghĩa agent
│   └── schemas.py              # Pydantic model
├── tools/                      # API tìm kiếm hình ảnh (bài 14)
│   ├── freepik_tools.py
│   └── dataforseo_tools.py
├── pipeline.py                 # Pipeline 4 bước (bài 15)
├── db.py                       # Tầng database (bài 16)
├── cli.py                      # Giao diện CLI (bài 17)
├── chat.py                     # Giao diện chat (bài 18)
└── workspace_tools.py          # Tools cho thành viên team (bài 18)
```

**Mẹo:** Khi muốn sửa gì đó, dùng bảng trên để tìm đúng file. Ví dụ, để thay đổi hướng dẫn cho writer, mở `output/agents/builders.py` và sửa danh sách instructions của `writer_agent`.

## Bài tập

Gọi trực tiếp các workspace tools (không qua chat team) để khám phá workspace:

1. Dùng `get_article_details(article_id)` để lấy chi tiết một bài viết (chọn một ID từ danh sách phía trên)
2. Parse kết quả JSON và in ra topic, status và số từ
3. Nếu bài viết có nội dung, dùng `get_article_content(article_id)` để đọc 500 ký tự đầu tiên

Đây chính xác là những gì Status Tracker agent làm khi bạn hỏi "Trạng thái bài viết số 3 thế nào?" — bạn đang gọi đúng những hàm mà nó gọi.

In [ ]:
# Bài tập: Viết code của bạn ở đây


## Hoàn thành Mô-đun 5!

Bạn đã hiểu **toàn bộ sản phẩm** — từ lưu trữ database đến lệnh CLI đến đội AI hội thoại.

Đây là những gì bạn đã học:

### Mô-đun 1: Nền tảng Python (Bài 1-4)
Biến, list, dictionary, hàm, package — nền móng cơ bản.

### Mô-đun 2: Hiểu về AI (Bài 5-7)
Cách LLM hoạt động, prompt và ngữ cảnh, lựa chọn model — cái "tại sao" đằng sau mọi thứ.

### Mô-đun 3: Xây dựng Agent (Bài 8-12)
Agent đầu tiên, tools, structured output, nối chuỗi, mini pipeline — thực hành xây dựng.

### Mô-đun 4: Pipeline SEO thực tế (Bài 13-15)
Agent nghiên cứu, dàn ý, viết bài, hình ảnh — pipeline sản xuất.

### Mô-đun 5: Sản phẩm hoàn chỉnh (Bài 16-18)
Database, CLI, giao diện chat — mọi thứ kết nối với nhau.

---

### Tiếp theo: Mô-đun 6 — Phát triển với sự hỗ trợ của AI

Bạn đã hiểu toàn bộ hệ thống. Nhưng mục tiêu không bao giờ là bắt bạn viết code từ đầu. Trong mô-đun cuối, bạn sẽ học cách dùng **Claude Code** — một trợ lý AI đọc hiểu codebase và thực hiện thay đổi cho bạn. Bạn sẽ học cách:
- **Chỉ đạo AI xây dựng tính năng** thay vì tự viết code
- **Kiểm tra thay đổi** bằng mọi thứ bạn đã học từ Mô-đun 1-5
- **Mở rộng sản phẩm** với agent, tools và tính năng mới

Đây là siêu năng lực thực sự: hiểu hệ thống đủ sâu để hướng dẫn AI cải thiện nó.